In [ ]:
import os
from autogen.agentchat import ConversableAgent
import sys
sys.path.append(os.path.abspath(".."))

from llm_config import llm_config

In [ ]:
job_advisor  = ConversableAgent(
    name = "JobAdvisor", #if user input is accepted, no LLM is required
    llm_config= llm_config,
    human_input_mode= "NEVER",
    max_consecutive_auto_reply= 1
)

In [ ]:
#Agent 1: Suggest relevant job roles
role_suggestor = ConversableAgent(
    name = "RoleSuggestor",
    system_message= "You are an expert in career coaching. Suggest 2-3 job roles based on user skills",
    llm_config= llm_config,
    human_input_mode="NEVER"
)

In [ ]:
#Agent 2: Check market demand
market_analyst = ConversableAgent(
    name= "MarketAnalyst",
    system_message= "You analyze job market trends. For a given role, respond with 'High', 'Moderate' or 'low' demand with the reason",
    llm_config= llm_config
)

In [ ]:
#Agent 3: Estimate salary
salary_estimator = ConversableAgent(
    name = "SalaryEstimator",
    system_message="Estimate an average annual salary for a given job role in USD.",
    llm_config=llm_config
)

In [ ]:
nested_chats = [
    {
        "recipient": role_suggestor,
        "message": "Suggest 2-3 job roles based on my skills in python, data analysis, and machine learning",
        "summary_method": "reflection_with_llm",
        "max_turns" : 1
    },
    {
        "recipient": market_analyst,
        "message": "Analyse the market demand for those job roles",
        "summary_method": "reflection_with_llm",
        "max_turns" : 1
    },
    {
        "recipient": salary_estimator,
        "message": "Estimate the average salary for those job roles",
        "summary_method": "reflection_with_llm",
        "max_turns" : 1
    }
]

In [ ]:
#Link nested logic to the Job Advisor
job_advisor.register_nested_chats(
    nested_chats,
    trigger= lambda sender: sender is None or sender.name not in ["RoleSuggestor", "MarketAnalyst","SalaryEstimator"]
)

In [ ]:
#Try with user input
reply = job_advisor.generate_reply(
    messages=[{'role': "user", "content": "I have skills in python, data analysis, and machine learning. What job should I go for?"}]
)

print("\nFinal recommendation", reply)